# Notebook 1 - Chatbot vs. Agent

## Why this notebook exists

You have used a chatbot. You type a question, you get an answer, you type again. It never *does* anything on its own.

An agent is different: it can break a task into steps, decide to use a tool, look at the result, and keep going until it is actually done. That difference is the whole workshop. Everything else (tools, retrieval, memory) hangs off it.

By the end of this notebook you will be able to **explain why an agent behaves differently from a chatbot**, because you will have built both and watched them run on the same task, side by side.

## What you'll build
1. A **chatbot**: one prompt in, one answer out.
2. An **agent**: the same task, but running a **plan -> act -> observe** loop you can read line by line.
3. A **side-by-side comparison** so the difference is visible, not asserted.

## Prerequisites
- A Google account (For Google Colab, no installation).
- Optional: an Anthropic **or** OpenAI API key. You don't need one to start. The notebook runs in `"mock"` mode with canned replies so you can see the mechanics first, then switch to a real model.
- For a live run, set `PROVIDER` to `openai` or `anthropic` and rerun from the top. Model name or provider may change.

One term, used consistently all workshop: an **agent** is a language model wrapped in a loop that lets it call tools and decide when it is finished.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ms-cc-org/AGENTIC-AI-Workshop/blob/main/notebooks/01_chatbot_vs_agent.ipynb)

## Step 1 - Setup

Run the cell below. Read it. This same adapter appears in every notebook. It is the seam that lets you swap Anthropic for OpenAI by changing one word.

**Don't worry if this looks long.**

This cell simply lets the notebook work with different AI providers. 
Throughout the workshop we'll treat it as a reusable utility and focus on the agent logic instead.


In [ ]:
#  SETUP cell. This is for a provider-agnostic LLM client.
#  Read this cell; every notebook uses this same adapter.
#  Switch providers by changing PROVIDER.
#  Nothingelse in the notebook changes. an agent is a pattern, not a vendor.

# In Google Colab this cell installs the packages. Locally, run once.
try:
    import google.colab
    !pip -q install anthropic openai
except Exception:
    pass

import os, json

PROVIDER = "openai"   # "anthropic" | "openai" | "mock"
                      # "mock" runs this notebook with NO API key, 

MODELS = {
    "anthropic": "claude-haiku-4-5-20251001",  # cheapest current Claude model
    "openai":    "gpt-5.4-nano", # cheapest current GPT model
}

# API keys
# In Colab: click the key icon (left sidebar) -> add ANTHROPIC_API_KEY or
# OPENAI_API_KEY, then uncomment the line below and your specific provider to load them.
#from google.colab import userdata
#os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
#os.environ["OPENAI_API_KEY"] = userdata.get("MY_API_KEY")

# Normalized shapes we use everywhere (so the agent code never mentions a vendor):
#   message: {"role":"user","content":str}
#            {"role":"assistant","content":str|None,"tool_calls":[{id,name,args}]}
#            {"role":"tool","tool_call_id":id,"name":name,"content":str}
#   tool:    {"name":str,"description":str,"parameters":<json-schema>}
#   reply:   {"text":str,"tool_calls":[{id,name,args}],"stop_reason":str}


_MOCK_SCRIPT = []
def mock_reset(script):
    global _MOCK_SCRIPT; _MOCK_SCRIPT = list(script)
def _mock_call(messages, tools):
    if _MOCK_SCRIPT:
        kind, payload = _MOCK_SCRIPT.pop(0)
        if kind == "tool":
            return {"text":"", "tool_calls":[{"id":"mock_"+payload["name"],
                    "name":payload["name"], "args":payload["args"]}], "stop_reason":"tool_use"}
        return {"text":payload, "tool_calls":[], "stop_reason":"end"}
    last = next((m for m in reversed(messages) if m["role"] in ("user","tool")), {"content":""})
    return {"text": f"[mock reply to] {str(last.get('content',''))[:80]}",
            "tool_calls":[], "stop_reason":"end"}


def _to_anthropic(messages):
    out=[]
    for m in messages:
        if m["role"]=="user":
            out.append({"role":"user","content":m["content"]})
        elif m["role"]=="assistant":
            blocks=[]
            if m.get("content"): blocks.append({"type":"text","text":m["content"]})
            for tc in m.get("tool_calls",[]):
                blocks.append({"type":"tool_use","id":tc["id"],"name":tc["name"],"input":tc["args"]})
            out.append({"role":"assistant","content":blocks})
        elif m["role"]=="tool":
            out.append({"role":"user","content":[{"type":"tool_result",
                        "tool_use_id":m["tool_call_id"],"content":str(m["content"])}]})
    return out

def _to_openai(messages, system):
    out=[]
    if system: out.append({"role":"system","content":system})
    for m in messages:
        if m["role"]=="user":
            out.append({"role":"user","content":m["content"]})
        elif m["role"]=="assistant":
            msg={"role":"assistant","content":m.get("content") or None}
            if m.get("tool_calls"):
                msg["tool_calls"]=[{"id":tc["id"],"type":"function","function":{
                    "name":tc["name"],"arguments":json.dumps(tc["args"])}} for tc in m["tool_calls"]]
            out.append(msg)
        elif m["role"]=="tool":
            out.append({"role":"tool","tool_call_id":m["tool_call_id"],"content":str(m["content"])})
    return out

def call_llm(messages, tools=None, system=None, max_tokens=1024, temperature=0):
    '''One function. Any provider. This is portable across whatever API key you have or your institution has.'''
    if PROVIDER=="mock":
        return _mock_call(messages, tools)
    if PROVIDER=="anthropic":
        from anthropic import Anthropic
        client=Anthropic()
        kw=dict(model=MODELS["anthropic"], max_tokens=max_tokens,
                temperature=temperature, messages=_to_anthropic(messages))
        if system: kw["system"]=system
        if tools: kw["tools"]=[{"name":t["name"],"description":t["description"],
                                "input_schema":t["parameters"]} for t in tools]
        r=client.messages.create(**kw)
        text=""; calls=[]
        for b in r.content:
            if b.type=="text": text+=b.text
            elif b.type=="tool_use": calls.append({"id":b.id,"name":b.name,"args":b.input})
        return {"text":text,"tool_calls":calls,"stop_reason":r.stop_reason}
    if PROVIDER=="openai":
        from openai import OpenAI
        client=OpenAI()
        kw=dict(model=MODELS["openai"], messages=_to_openai(messages, system),
                temperature=temperature, max_completion_tokens=max_tokens)
        if tools: kw["tools"]=[{"type":"function","function":{"name":t["name"],
                    "description":t["description"],"parameters":t["parameters"]}} for t in tools]
        r=client.chat.completions.create(**kw)
        msg=r.choices[0].message
        calls=[{"id":tc.id,"name":tc.function.name,
                "args":json.loads(tc.function.arguments or "{}")} for tc in (msg.tool_calls or [])]
        return {"text":msg.content or "", "tool_calls":calls,
                "stop_reason":"tool_use" if calls else "end"}
    raise ValueError("Unknown PROVIDER: "+PROVIDER)

print(f"Setup ready. PROVIDER={PROVIDER!r}.")

![](../Images/Chatbot-vs-Agent.jpg)

## Step 2 — The chatbot

A chatbot is a **single call** to a language model. Prompt goes in, text comes out, and the model has no way to take an action in the world.

That is powerful, but it is *one shot*. If the task needs a real fact the model doesn't have, or a calculation it can't do reliably in its head, a chatbot will guess. Watch it do exactly that.

In [ ]:
ABSTRACTS = {
    "A": ("While large language model (LLM)-based chatbots excel at handling linear, single-turn customer inquiries, they frequently fail when faced with complex, multi-layered user objectives requiring cross-departmental resolution."
          "We propose **MetaCoord**, a hierarchical multi-agent framework designed to orchestrate specialized chatbot agents. The architecture utilizes a centralized coordinator agent that applies Chain-of-Thought (CoT) prompting to decompose ambiguous user queries into sequential sub-tasks. These sub-tasks are then routed to domain-specific specialist agents (e.g., billing, technical troubleshooting, and logistics) equipped with localized Retrieval-Augmented Generation (RAG) pipelines. A shared state-machine memory buffer ensures context retention across agent transitions."
          "We evaluated MetaCoord against a single-agent GPT-4 baseline using a newly curated dataset of 5,000 multi-turn, synthetic customer service interactions mimicking complex retail disputes. Performance was measured via Task Success Rate (TSR), Average Turns to Resolution (ATR), and semantic hallucination rates."
          "MetaCoord achieved a **TSR of 89.4%**, representing a 14.2percent absolute improvement over the single-agent baseline. Furthermore, the framework reduced the ATR from 6.4 to 4.2 turns and lowered hallucination rates by 38%, demonstrating the efficacy of localized, modular knowledge boundaries in multi-agent ecosystems."),
    "B": ("Conversational agents deployed in healthcare settings often prioritize factual accuracy at the expense of patient rapport, leading to low user compliance and poor long-term engagement."
          "This paper introduces **AffectiveAgent**, a healthcare chatbot framework that integrates a real-time sentiment analysis loop with a Reinforcement Learning from Human Feedback (RLHF) model specifically tuned for empathetic alignment. The system extracts valence and arousal indices from user text inputs and dynamically adjusts the decoding temperature and prompt engineering strategies of the underlying LLM. This allows the chatbot to modulate its tone between clinical authority and empathetic support depending on user distress levels."
          "A randomized controlled trial was conducted with 350 simulated user profiles interacting with the chatbot regarding chronic symptom tracking over a 14-day period. Evaluation metrics included the Working Alliance Inventory (WAI) score, linguistic alignment, and daily user compliance rates."
          "AffectiveAgent demonstrated a statistically significant increase in perceived empathy, scoring **22 percent higher on the WAI** compared to standard, non-affective baseline models. Crucially, this emotional alignment led to a 15.8 percent improvement in daily user data-logging compliance without any statistically significant degradation in the clinical accuracy of the medical information provided."),
    "C": ("Deploying autonomous AI agents capable of continuous tool-use and reasoning on edge devices remains bottlenecked by high computational latency and memory constraints."
          "We present **DistillAgent**, a novel methodology for deploying high-capability autonomous chatbots on resource-constrained hardware. The framework utilizes a cascaded architecture featuring a highly distilled 3-billion (3B) parameter student model running locally for routine dialogue and initial intent classification. For complex tool-calling and reasoning steps, the framework employs speculative decoding, using the local model to draft token sequences that are verified by a cloud-hosted 70-billion (70B) parameter teacher model only when local confidence drops below a dynamic threshold ($\tau = 0.75$)."
          "The system was benchmarked on the AgentBench suite and deployed on commercial edge hardware (Raspberry Pi 5 and Jetson Orin Nano). We evaluated inference latency, memory footprint, cloud API cost reductions, and task execution accuracy."
          "Empirical results demonstrate that DistillAgent achieves a **4.1x reduction in end-to-end inference latency** and reduces cloud infrastructure costs by 73.2 percent compared to fully cloud-reliant agent architectures. The local memory footprint remained stably under 4.5 GB of RAM, while retaining 94.6 percent of the task completion accuracy of the full, uncompressed cloud model."),
}

def chatbot(prompt, system=None):
    #One call. One answer. No tools, no loop.
    return call_llm([{"role":"user","content":prompt}], system=system)["text"]

all_three = "\n\n".join(ABSTRACTS.values())
# Uncomment this part if you're executing on mock provider mode
'''
mock_reset([("final", 
    "This is a MOCK response." 
    "Summary:  All three study LLMs for literature review. Paper A screens abstracts with "
    "high recall but needs human checks; Paper B speeds up summarizing but makes citation "
    "errors; Paper C uses multiple agents for better coverage at higher cost. ")])
'''

print("CHATBOT answer:\n", chatbot("Summaroze these three abstracts:\n\n" + all_three))

The chatbot did fine, but notice its limits: **you** had to gather and paste every abstract, it produced prose rather than a comparison you can scan, and it had no way to go read a paper it wasn't handed. It can only work with what's already in the prompt.

That is the gap an agent closes.

## Step 3 — The agent

An agent is the same model, but wrapped in a loop and handed some tools. Each turn it can either **call a tool** or **give a final answer**. When it calls a tool, we run the real function, feed the result back, and let it continue.

This pattern is called **ReAct** (Reason + Act): the model reasons about what to do, acts by calling a tool, observes the result, and repeats.

user task --> model
if (no tool call) --> final answer
if (tool call) --> run the tool --> observe (feed result back) --> loop again


In [ ]:
def run_agent(user_task, tools_spec, tool_fns, system=None, max_steps=6, verbose=True):
    '''The agent loop. The idea is:
       plan -> act (maybe call a tool) -> observe (feed the result back) -> repeat,
       until the model returns a final answer instead of a tool call.'''
    messages = [{"role":"user","content":user_task}]
    for step in range(1, max_steps+1):
        reply = call_llm(messages, tools=tools_spec, system=system)
        if reply["tool_calls"]:                                  # the model wants a tool
            messages.append({"role":"assistant","content":reply["text"],
                             "tool_calls":reply["tool_calls"]})
            for tc in reply["tool_calls"]:
                if verbose: print(f"  step {step}: tool `{tc['name']}` <- {tc['args']}")
                try:    result = tool_fns[tc["name"]](**tc["args"])   # run the real function
                except Exception as e: result = f"ERROR: {e}"
                messages.append({"role":"tool","tool_call_id":tc["id"],
                                 "name":tc["name"],"content":result})   # observe
        else:                                                    # no tool -> we are done
            if verbose: print(f"  step {step}: final answer")
            return reply["text"], messages
    return "Stopped: hit max_steps.", messages

### Give the agent one tool: reading a paper

A **tool** is just a Python function plus a small description that tells the model that it exists and what arguments it takes. Here the agent can fetch abstract of any paper from the library, one at a time, the way we pull a paper one by one while doing a review.

In [ ]:
def read_abs(paper):
    #return the abstract text for a paper with ID (A,B or C)
    return ABSTRACTS.get(paper.strip().upper(), f"(There is no paper '{paper}' in the library)")

def make_tools():
    ids = ", ".join(ABSTRACTS)
    return [
    {"name": "read_abstract",
     "description": f"Read the abstract of one paper from the review library. "
                        f"Call once per a paper you need. Valid Ids: {ids}.",
     "parameters": {"type": "object", "properties": {
         "paper": {"type": "string", "description": "paper id, e.g. 'A'"}},
         "required": ["paper"]}},
    ]

tool_specs = make_tools()
tool_fns = {"read_abstract": read_abs}
print("1 tool registered: ", list(tool_fns), "| library has papers:", list(ABSTRACTS))

In [ ]:
# Script the agent's decisions for mock mode: read papers and producing a table.
# (With a real provider, the MODEL makes these decisions itself. Make sure Set PROVIDER above.)

comparision_table = (
 "| paper | method | main findings | key limitation |\n"
 "|---|---|---|---|\n"
 "| A | LLM Abstract screener | High recall .95 | Low on Precision |\n"
 "| B | Agentic AI | 50 percent drafting | 12 percent citation errors |\n"
 "| C | Multi-agent pipelines | Btter coverage | Higher cost and harder to debug |\n") 

mock_reset([
    ("tool", {"name": "read_abs", "args": {"paper": "A"}}),
    ("tool", {"name": "read_abs", "args": {"paper": "B"}}),
    ("tool", {"name": "read_abs", "args": {"paper": "C"}}),
    ("final", comparision_table),
])

print("AGENT run:")
answer, transcript = run_agent(
    "Do a literature review of papers A, B, C. read each one and compare their findings and produce a comparision table",
    tool_specs, tool_fns)
print("\nAGENT answer:\n", answer)

## Step 4 — Side by side

Notice the difference in *behaviour*, not just wording.

In [ ]:
print("|."*50)
print("CHATBOT  : one call, no way to check facts -> guesses.")
print("AGENT    : loops, calls `today`, then `days_between` -> exact answer.")
print("|."*50)

print("\nThe agent's transcript (what actually happened in each turn):")
for m in transcript:
    role = m["role"]
    if role == "assistant" and m.get("tool_calls"):
        for tc in m["tool_calls"]:
            print(f"  assistant -> read paper {tc['args'].get('paper')}")
    elif role == "tool":
        print(f"  tool -> returned {len(str(m['content']))} chars of abstract")
    elif role == "user":
        print(f"  user -> (review task)")

The **chatbot** needed you to assemble the inputs. The agent **gathered them itself**, one paper at a time and chose the output shape as well. On three papers, that can be considered as a small win. On thirty, it is the difference of hours from your afternoon to coffee break.

## The anatomy of an agent

Every agent you will ever meet is some arrangement of these five parts. You just built the first three.

- **A model** - the language model that reasons and decides. (the `call_llm` seam)
- **A loop** - turns one answer into many steps. (`run_agent`)
- **Tools** - functions that let it act in the world. 
- **Memory** - context kept across steps. 
- **A stopping condition** - how it knows it's done. (here: "no tool call = done", plus `max_steps` as a safety net)

## Exercise

Before running anything, think like a reviewer:

**Imagine you're preparing a literature review. What information would an agent need to gather before it can write the summary?**

Writing down a short list (the abstract of each paper? the methods? the sample sizes? the venue and year? conflicts of interest?). That list *is* the agent's plan.

Now make it real:
1. Add a **fourth paper** to `ABSTRACTS` below (either real or synthetic).
2. Ask the agent to include it in the review.
3. If you're in `"mock"` mode, add a `read_abstract` step for the new paper to the script. If you've set a real `PROVIDER` and key, delete the `mock_reset(...)` line and let the model decide what to read.
4. Run it and compare the agent's reading order to the list you wrote.

**Question to answer for yourself:** which step did the agent do that the chatbot *couldn't*, no matter how good its summary was?

In [ ]:
# Add your fourth paper:
ABSTRACTS["D"] = ("Paper D - (your abstract here). Example: fine-tuning a small open model "
                  "for screening matched the large model's recall at lower cost, but needed "
                  "500 labeled examples to get there.")

tool_specs = make_tools()

# Run the review including D. (Comment out mock_reset line once PROVIDER is real.)
mock_reset([
    ("tool", {"name":"read_abstract", "args":{"paper":"A"}}),
    ("tool", {"name":"read_abstract", "args":{"paper":"B"}}),
    ("tool", {"name":"read_abstract", "args":{"paper":"C"}}),
    ("tool", {"name":"read_abstract", "args":{"paper":"D"}}),
    ("final", "Updated table would add Paper D: fine-tuned small model, matches recall at "
              "lower cost, but needs ~500 labeled examples. (See the read steps above.)"),
])

ans, tr = run_agent("Review papers A, B, C, and D. Read each, then compare.", tool_specs, tool_fns)
reads = [m["tool_calls"][0]["args"]["paper"] for m in tr if m["role"]=="assistant" and m.get("tool_calls")]
print("The agent chose to read, in order:", reads)
print("\nResult:\n", ans)

## Workflows vs. agents

- **Workflow:** you, the developer, fix the steps in advance (call model, then search, then summarize). Predictable, cheaper, easier to debug.
- **Agent:** the *model* decides the steps at run time. Flexible, but more calls, more cost, harder to predict.

Our literature review is mostly a workflow with a little agency (the model still decides how to compare and what the table says). Most real research tools live right here.

**Cost is the honest catch.** A chatbot is one API call. An agent that takes 4 tool-steps is at least 5 calls, and each call resends the growing transcript. So an agent can cost several times what a chatbot costs for the same question. Use the loop where the flexibility earns its price, **not everywhere**.

In [ ]:
# Rough cost intuition (not a benchmark): count model calls in the run above.
calls = sum(1 for m in transcript if m["role"]=="assistant")
print(f"This agent run made ~{calls}+ model calls vs. 1 for the chatbot.")
print("Rule of thumb: reach for the loop when the steps can't be known in advance.")

## Reflect and Discuss

- When is a **chatbot** enough?
- When do you actually need the **loop**? (multi-step, needs a real fact, needs to *do* something)
- Where in *your* research, what would you want an agent to "go find out and come back" save you time and writes a word?